# 01a — Preprocess: qasim21 Video Stream Person Detection

Downloads and preprocesses the [Video Stream Dataset for YOLOv5/v7](https://www.kaggle.com/datasets/qasim21/video-stream-dataset-for-yolov5v7-for-detection).

Video stream frames annotated for person detection, YOLO format.

| | |
|---|---|
| **Source** | `qasim21/video-stream-dataset-for-yolov5v7-for-detection` |
| **Size** | ~39MB |
| **Classes** | `person` |
| **Format** | YOLO (TXT) |

In [ ]:
!pip install kaggle albumentations -q

import os, shutil, glob, random, yaml
import numpy as np
import matplotlib.pyplot as plt
import cv2
from collections import Counter

print('Setup done!')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/AI_TRAINING/GreenVision'
OUTPUT_DIR = '/content/dataset_qasim21'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')

## 1. Download Dataset

In [ ]:
# Setup Kaggle credentials from Colab secret
import json, os
from google.colab import userdata

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)

# Read token from Colab secret
token = userdata.get('KAGGLE_API_TOKEN')

# Write kaggle.json
username = 'dngdngphmminh'  # your Kaggle username
creds = {"username": username, "key": token}

with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
    json.dump(creds, f)
os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)
print('Kaggle credentials configured.')

In [ ]:
RAW_DIR = '/content/raw_qasim21'

if not os.path.exists(RAW_DIR):
    print('Downloading qasim21/video-stream-dataset-for-yolov5v7-for-detection...')
    !kaggle datasets download -d qasim21/video-stream-dataset-for-yolov5v7-for-detection -p /content --unzip
    # Find the extracted folder
    import subprocess
    result = subprocess.run(['find', '/content', '-maxdepth', '2', '-type', 'd', '-name', '*qasim*', '-o', '-name', '*video*', '-o', '-name', '*stream*', '-o', '-name', '*yolo*'],
                           capture_output=True, text=True)
    candidates = [l for l in result.stdout.strip().split('\n') if l and l != '/content']
    if candidates:
        RAW_DIR = candidates[0]
    else:
        # Check common patterns
        for d in ['dataset', 'data', 'images', 'Video Stream dataset for Yolov5,v7 for Detection']:
            if os.path.exists(f'/content/{d}'):
                RAW_DIR = f'/content/{d}'
                break
    print(f'Extracted to: {RAW_DIR}')
else:
    print(f'Already extracted: {RAW_DIR}')

## 2. Inspect Raw Structure

In [ ]:
# Walk the directory tree to understand structure
print(f'Root: {RAW_DIR}')
print()
for root, dirs, files in os.walk(RAW_DIR):
    depth = root.replace(RAW_DIR, '').count(os.sep)
    indent = '  ' * depth
    folder = os.path.basename(root)
    print(f'{indent}{folder}/')
    if depth < 3:
        # Show file types and counts
        exts = Counter(os.path.splitext(f)[1].lower() for f in files)
        for ext, count in exts.most_common(10):
            print(f'{indent}  {ext or "(no ext)"}: {count} files')
        # Show sample filenames
        if files:
            samples = files[:3]
            for s in samples:
                print(f'{indent}    e.g. {s}')

In [ ]:
# Find images and label files
all_images = glob.glob(os.path.join(RAW_DIR, '**', '*.jpg'), recursive=True) + \
             glob.glob(os.path.join(RAW_DIR, '**', '*.png'), recursive=True) + \
             glob.glob(os.path.join(RAW_DIR, '**', '*.jpeg'), recursive=True)

all_labels = glob.glob(os.path.join(RAW_DIR, '**', '*.txt'), recursive=True)

# Filter out README/data.yaml files
all_labels = [f for f in all_labels if not os.path.basename(f).lower().startswith(('readme', 'classes', 'data', 'train', 'val', 'test'))]

print(f'Images found: {len(all_images)}')
print(f'Label files found: {len(all_labels)}')

# Check if labels are YOLO format
if all_labels:
    sample_label = open(all_labels[0]).read().strip()
    print(f'\nSample label ({os.path.basename(all_labels[0])}):')
    print(sample_label[:300])

# Check for data.yaml
yaml_files = glob.glob(os.path.join(RAW_DIR, '**', '*.yaml'), recursive=True) + \
             glob.glob(os.path.join(RAW_DIR, '**', '*.yml'), recursive=True)
if yaml_files:
    print(f'\nFound YAML config:')
    print(open(yaml_files[0]).read())

## 3. Convert to Standard YOLO Format

Normalizes the dataset into `images/` and `labels/` with YOLO annotations. Remaps all classes to `0: person`.

In [ ]:
import yaml as yaml_lib

# Clean output dir
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(f'{OUTPUT_DIR}/images', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/labels', exist_ok=True)

# Strategy 1: Already has images/ + labels/ structure with matching names
# Strategy 2: Images and labels in same directory
# Strategy 3: Nested train/val structure

def match_image_label(img_path, label_dirs):
    """Find the matching label file for an image."""
    base = os.path.splitext(os.path.basename(img_path))[0]
    for ldir in label_dirs:
        candidate = os.path.join(ldir, base + '.txt')
        if os.path.exists(candidate):
            return candidate
    # Check same directory as image
    candidate = os.path.join(os.path.dirname(img_path), base + '.txt')
    if os.path.exists(candidate):
        return candidate
    return None

# Collect all possible label directories
label_dirs = set()
for lbl in all_labels:
    label_dirs.add(os.path.dirname(lbl))
label_dirs = list(label_dirs)

paired = []
no_label = 0

for img_path in all_images:
    lbl_path = match_image_label(img_path, label_dirs)
    if lbl_path and os.path.exists(lbl_path):
        paired.append((img_path, lbl_path))
    else:
        no_label += 1

print(f'Paired (image + label): {len(paired)}')
print(f'Images without labels: {no_label}')

# Copy to output dir, remap all classes to 0 (person)
copied = 0
skipped = 0

for img_path, lbl_path in paired:
    # Read label
    with open(lbl_path) as f:
        lines = f.read().strip().split('\n')

    yolo_lines = []
    for line in lines:
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        # Remap any class to 0 (person)
        cx, cy, bw, bh = parts[1], parts[2], parts[3], parts[4]
        yolo_lines.append(f'0 {cx} {cy} {bw} {bh}')

    if not yolo_lines:
        skipped += 1
        continue

    base = os.path.splitext(os.path.basename(img_path))[0]
    ext = os.path.splitext(img_path)[1]

    # Handle duplicate names
    out_img = f'{OUTPUT_DIR}/images/{base}{ext}'
    out_lbl = f'{OUTPUT_DIR}/labels/{base}.txt'
    if os.path.exists(out_img):
        base = base + '_' + str(random.randint(1000, 9999))
        out_img = f'{OUTPUT_DIR}/images/{base}{ext}'
        out_lbl = f'{OUTPUT_DIR}/labels/{base}.txt'

    shutil.copy2(img_path, out_img)
    with open(out_lbl, 'w') as f:
        f.write('\n'.join(yolo_lines))
    copied += 1

print(f'\nCopied {copied} image-label pairs')
print(f'Skipped (empty labels): {skipped}')

## 4. Augment (3x)

In [ ]:
import albumentations as A

img_dir = f'{OUTPUT_DIR}/images'
lbl_dir = f'{OUTPUT_DIR}/labels'

imgs = glob.glob(os.path.join(img_dir, '*.jpg')) + glob.glob(os.path.join(img_dir, '*.png'))
print(f'Original images: {len(imgs)}')

aug_flip = A.Compose([
    A.HorizontalFlip(p=1.0),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

aug_bc = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.2, p=1.0),
    A.GaussNoise(var_limit=(10, 30), p=0.7),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

augmented = 0

for img_path in imgs:
    base = os.path.splitext(os.path.basename(img_path))[0]
    ext = os.path.splitext(img_path)[1]
    lbl_path = os.path.join(lbl_dir, base + '.txt')

    if not os.path.exists(lbl_path):
        continue

    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

    with open(lbl_path) as f:
        lines = f.read().strip().split('\n')
    bboxes, labels = [], []
    for line in lines:
        parts = line.strip().split()
        if len(parts) == 5:
            labels.append(int(parts[0]))
            bboxes.append([float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])])

    if not bboxes:
        continue

    # Horizontal flip
    try:
        result = aug_flip(image=img, bboxes=bboxes, class_labels=labels)
        cv2.imwrite(os.path.join(img_dir, f'{base}_flip{ext}'),
                    cv2.cvtColor(result['image'], cv2.COLOR_RGB2BGR))
        with open(os.path.join(lbl_dir, f'{base}_flip.txt'), 'w') as f:
            for cls, bb in zip(result['class_labels'], result['bboxes']):
                f.write(f'{cls} {bb[0]:.6f} {bb[1]:.6f} {bb[2]:.6f} {bb[3]:.6f}\n')
        augmented += 1
    except Exception:
        pass

    # Brightness + contrast + noise
    try:
        result = aug_bc(image=img, bboxes=bboxes, class_labels=labels)
        cv2.imwrite(os.path.join(img_dir, f'{base}_bc{ext}'),
                    cv2.cvtColor(result['image'], cv2.COLOR_RGB2BGR))
        with open(os.path.join(lbl_dir, f'{base}_bc.txt'), 'w') as f:
            for cls, bb in zip(result['class_labels'], result['bboxes']):
                f.write(f'{cls} {bb[0]:.6f} {bb[1]:.6f} {bb[2]:.6f} {bb[3]:.6f}\n')
        augmented += 1
    except Exception:
        pass

final_imgs = glob.glob(os.path.join(img_dir, '*.jpg')) + glob.glob(os.path.join(img_dir, '*.png'))
print(f'\nAugmented: +{augmented} images')
print(f'Total: {len(final_imgs)} images ({len(final_imgs)/len(imgs):.1f}x)')

## 5. Stats & Preview

In [ ]:
img_dir = f'{OUTPUT_DIR}/images'
lbl_dir = f'{OUTPUT_DIR}/labels'

imgs = glob.glob(os.path.join(img_dir, '*.*'))
lbls = glob.glob(os.path.join(lbl_dir, '*.txt'))

total_boxes = 0
for lbl in lbls:
    with open(lbl) as f:
        total_boxes += len(f.readlines())

print('=' * 45)
print('  qasim21 Dataset Summary')
print('=' * 45)
print(f'  Images:        {len(imgs)}')
print(f'  Labels:        {len(lbls)}')
print(f'  Bounding boxes: {total_boxes}')
print(f'  Avg boxes/img:  {total_boxes/len(lbls):.1f}')
print('=' * 45)

# Sample visualization
CLASS_COLOR = (0, 255, 0)

originals = [p for p in imgs if '_flip' not in os.path.basename(p) and '_bc' not in os.path.basename(p)]
sample = random.sample(originals, min(12, len(originals)))

fig, axes = plt.subplots(3, 4, figsize=(20, 13))
for ax, img_path in zip(axes.flatten(), sample):
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    base = os.path.splitext(os.path.basename(img_path))[0]
    lbl = os.path.join(lbl_dir, base + '.txt')
    count = 0
    if os.path.exists(lbl):
        with open(lbl) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cx, cy, bw, bh = map(float, parts[1:5])
                    x1 = int((cx - bw/2) * w)
                    y1 = int((cy - bh/2) * h)
                    x2 = int((cx + bw/2) * w)
                    y2 = int((cy + bh/2) * h)
                    cv2.rectangle(img, (x1, y1), (x2, y2), CLASS_COLOR, 2)
                    count += 1
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'{count} person(s)', fontsize=9)
plt.suptitle('qasim21 — Video Stream Person Detection', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Save to Drive

In [ ]:
# Save preprocessed dataset to Drive
drive_output = os.path.join(DRIVE_ROOT, 'datasets', 'qasim21')
if os.path.exists(drive_output):
    shutil.rmtree(drive_output)
shutil.copytree(OUTPUT_DIR, drive_output)

print(f'Saved to: {drive_output}')
print(f'Images: {len(glob.glob(os.path.join(drive_output, "images", "*.*")))}')
print(f'Labels: {len(glob.glob(os.path.join(drive_output, "labels", "*.txt")))}')

---
## Done!

Preprocessed dataset saved to:
```
/content/drive/MyDrive/AI_TRAINING/GreenVision/datasets/qasim21/
  images/   — all images (original + augmented)
  labels/   — YOLO format labels (class 0 = person)
```

Ready to merge with other datasets or use for training.